# recs_011 — Candidate retrieval comparison (same offline contract)

**Purpose.** Hold **one offline contract** (same Task A examples, slices, personalization) and **compare candidate methods** side by side—not to train a new tower here.
- **Baselines:** bi-encoder / heuristic user-item similarity from the central job (`run_retrieval_eval` + `recs_job_eval_retrieval.py`).
- **Candidates:** Locked **A/B/C/D/E query-side recipes** below (fixed encoder + fusion rules today). Names use `two_tower_*` as **experiment IDs**; this notebook evaluates those candidates, optionally plus **precomputed CSVs** if a model was trained **elsewhere**.

## Decision metrics (read this first)

**Config cutoffs** (defaults in `configs/recs_job_eval_retrieval.json`): **`k_retrieval`** drives retrieval-stage Hit/Precision/Recall; **`k_final`** drives ranking-stage MAP/NDCG/MRR and the ranking-table Hit/Precision/Recall; **`k_personalization`** drives short-list guardrails (`CatalogCoverage@{k_personalization}`, `ILD@{k_personalization}`, …). CSV columns stay named `Hit@K` / `NDCG@K` — **K is symbolic**; the integer depth is whichever cutoff applies to that artifact row.

Authoritative write-up: **`docs/eval_contract.md`**.

| Stage | Slice | Cohort (`n_eval_targets`) | Primary | Secondary | Tie-break / next |
|-------|-------|---------------------------|---------|-----------|------------------|
| Retrieval | A | ≥ 2 | Recall@K (`k_retrieval`) | Precision@K | Hit@K |
| Retrieval | B | 1 | Hit@K | Precision@K | Recall@K |
| Ranking | A | ≥ 2 | NDCG@K (`k_final`) | MAP@K | MRR, then Hit / Precision / Recall |
| Ranking | B | 1 | Hit@K (`k_final`) | MRR | NDCG@K, MAP@K, … |

Slice C (`n_eval_targets == 0`): contract uses coverage-only / no relevance gate; this notebook focuses on slice A/B candidates.

**Framework note:** the eval job intentionally splits **retrieval** vs **ranking** summaries plus per-example JSONL. It is verbose; the upside is no accidental mixing of **top-100 recall** with **top-10 ranking** unless you ignore the table above.

## Candidate definitions (locked)

- **Candidate A (`two_tower_a_raw_text`)**
  - Query/user tower input: current raw review text only (`q_session`).
- **Candidate B (`two_tower_b_raw_plus_mean_train`)**
  - Query/user tower input: current raw review text + `mean_train_history` (from past train reviews only).
- **Candidate C (`two_tower_c_raw_plus_behavior`)**
  - Query/user tower input: current raw review text + `u_behavior`.
- **Candidate D (`two_tower_d_raw_plus_habit`)**
  - Query/user tower input: current raw review text + `u_habit`, where `u_habit = normalize(0.5 * u_behavior + 0.5 * u_reviews)`.
- **Candidate E (`two_tower_e_pref_structured_session`)**
  - **Preference / structured session only** (parallel to A, no history fusion): heuristic `extract_preferences` → `PreferenceProfile`-backed dict → `build_embedding_input` → embed with the **same Hub model** as `q_session`. Ablation vs raw text; rules v0 often loses to raw on val (see `docs/recommender_transition_plan.md`); include to measure it in this fused grid.
- **Item tower (shared by A/B/C/D/E)**
  - Game/item text representation used for full-catalog retrieval.

## Playtime feature decisions (locked)

- **Query scalar features** (used with query/user features):
  - `log1p(query_playtime_hours)`
  - `log1p(author.playtime_last_two_weeks)`
- **History weighting for behavior**:
  - Build `u_behavior` as a playtime-weighted mean of historical game embeddings using train-history only.
  - Per-game weight: `w_i = log1p(hours_i)`.
  - Vector form: `u_behavior = sum(w_i * emb(game_i)) / sum(w_i)`.
- **Scope note**:
  - Apply playtime weighting to `u_behavior` only (not required for `u_reviews`).

This A–E grid tests incremental value of **structured preference text** vs raw session, plus review-history (`recs_004`), behavior-history, and habit fusion — item-side representation stays fixed.

**If you add a neural retriever later (outside this notebook).** Train + export checkpoints or precomputed **`eval_ranking_*.csv`-shaped tables** elsewhere; optionally add a loader that emits full-catalog scores per example so those runs join the **same metric code** here.

**Notebook role.** **Compare candidates** (tables, diagnostics, optional CSV ingest). Prefer mirroring `eval_ranking_*.csv` column shapes for apples-to-apples diffing; baseline job writes `eval_retrieval_*.csv` alongside.

## Prerequisites

- Run retrieval baseline job:
  - `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json`
- Artifacts (default): `artifacts/recs/offline_eval/runs/latest/` — `eval_ranking_*.csv`, `eval_retrieval_*.csv`, `eval_offline_run_meta.json`.
- **Optional:** precomputed candidate overall table `eval_ranking_two_tower_overall.csv` (same columns as `eval_ranking_overall.csv` for the candidate method). Legacy name `eval_retrieval_two_tower_overall.csv` also works if you rename or symlink.
- **Trained checkpoints (optional):** only if you wire them in separately; comparison itself uses baselines + in-notebook candidates and/or CSV drops.

In [29]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from steam_review_ml.recommender.evaluation import METRIC_COLS, RETRIEVAL_METRIC_COLS
from steam_review_ml.utils import load_config


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())

EVAL_JOB_CONFIG = REPO_ROOT / "configs" / "recs_job_eval_retrieval.json"
job_cfg = load_config(str(EVAL_JOB_CONFIG))
BASELINE_EVAL_DIR = REPO_ROOT / str(job_cfg.get("output_dir", "artifacts/recs/offline_eval/runs/latest"))

PATH_OVERALL = BASELINE_EVAL_DIR / "eval_ranking_overall.csv"
PATH_BY_SLICE = BASELINE_EVAL_DIR / "eval_ranking_by_slice.csv"
PATH_RETRIEVAL_OVERALL = BASELINE_EVAL_DIR / "eval_retrieval_overall.csv"
PATH_RETRIEVAL_BY_SLICE = BASELINE_EVAL_DIR / "eval_retrieval_by_slice.csv"
PATH_RUN_META = BASELINE_EVAL_DIR / "eval_offline_run_meta.json"

# When you precompute a two-tower eval with the same contract, drop a matching CSV here (optional).
PATH_TWO_TOWER_OVERALL = BASELINE_EVAL_DIR / "eval_ranking_two_tower_overall.csv"
_legacy_tt = BASELINE_EVAL_DIR / "eval_retrieval_two_tower_overall.csv"
if not PATH_TWO_TOWER_OVERALL.is_file() and _legacy_tt.is_file():
    PATH_TWO_TOWER_OVERALL = _legacy_tt

BASELINE_METHODS = ["raw", "popularity_train", "multi_mean_train"]
CANDIDATE_METHODS = [
    "two_tower_a_raw_text",
    "two_tower_b_raw_plus_mean_train",
    "two_tower_c_raw_plus_behavior",
    "two_tower_d_raw_plus_habit",
    "two_tower_e_pref_structured_session",
]

K_FINAL = int(job_cfg.get("k_final", 10))
K_RETRIEVAL = int(job_cfg.get("k_retrieval", K_FINAL))
K_PERSONALIZATION = int(job_cfg.get("k_personalization", K_FINAL))

# Display sort keys (evaluation policy): ranking overall / retrieval overall / within-slice.
SORT_RANKING_OVERALL = ["NDCG@K", "MAP@K", "MRR", "Hit@K", "Precision@K", "Recall@K"]
SORT_RETRIEVAL_OVERALL = ["Recall@K", "Precision@K", "Hit@K"]


def _sort_ranking_by_slice_for_display(df: pd.DataFrame) -> pd.DataFrame:
    """Within each cohort slice: apply slice-specific primary/secondary keys, then concatenate."""
    blocks: list[pd.DataFrame] = []
    specs: tuple[tuple[str, list[str]], ...] = (
        ("slice_a_multi_target", ["NDCG@K", "MAP@K", "MRR", "Hit@K", "Precision@K", "Recall@K"]),
        ("slice_b_single_target", ["Hit@K", "MRR", "NDCG@K", "MAP@K", "Precision@K", "Recall@K"]),
        ("slice_c_zero_target", ["NDCG@K", "MAP@K", "MRR", "Hit@K"]),
        ("slice_other", ["NDCG@K", "MAP@K", "MRR", "Hit@K"]),
    )
    known_slices = {s for s, _ in specs}
    for sl, cols in specs:
        sub = df[df["slice_name"] == sl]
        if sub.empty:
            continue
        use = [c for c in cols if c in sub.columns]
        blocks.append(sub.sort_values(use, ascending=False))
    tail = df[~df["slice_name"].isin(known_slices)]
    if not tail.empty:
        blocks.append(tail.sort_values(["slice_name", "NDCG@K"], ascending=[True, False]))
    return pd.concat(blocks, ignore_index=True) if blocks else df.copy()


def _sort_retrieval_by_slice_for_display(df: pd.DataFrame) -> pd.DataFrame:
    blocks: list[pd.DataFrame] = []
    specs: tuple[tuple[str, list[str]], ...] = (
        ("slice_a_multi_target", ["Recall@K", "Precision@K", "Hit@K"]),
        ("slice_b_single_target", ["Hit@K", "Precision@K", "Recall@K"]),
        ("slice_c_zero_target", ["Recall@K", "Hit@K", "Precision@K"]),
        ("slice_other", ["Recall@K", "Hit@K", "Precision@K"]),
    )
    known_slices = {s for s, _ in specs}
    for sl, cols in specs:
        sub = df[df["slice_name"] == sl]
        if sub.empty:
            continue
        use = [c for c in cols if c in sub.columns]
        blocks.append(sub.sort_values(use, ascending=False))
    tail = df[~df["slice_name"].isin(known_slices)]
    if not tail.empty:
        tail_cols = [c for c in ["Recall@K", "Hit@K"] if c in tail.columns]
        blocks.append(tail.sort_values(["slice_name"] + tail_cols, ascending=[True] + [False] * len(tail_cols)))
    return pd.concat(blocks, ignore_index=True) if blocks else df.copy()

In [30]:
for p in (PATH_OVERALL, PATH_BY_SLICE):
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {p}\n"
            "Run: python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json"
        )

baseline_overall = pd.read_csv(PATH_OVERALL)
baseline_by_slice = pd.read_csv(PATH_BY_SLICE)

run_meta = None
if PATH_RUN_META.is_file():
    run_meta = json.loads(PATH_RUN_META.read_text(encoding="utf-8"))

required = ["method", *METRIC_COLS]
missing = [c for c in required if c not in baseline_overall.columns]
if missing:
    raise ValueError(f"eval_ranking_overall.csv missing columns: {missing}")

baseline_overall_sub = baseline_overall[baseline_overall["method"].isin(BASELINE_METHODS)].copy()

print("Loaded baseline tables:")
print(" ", PATH_OVERALL, f"rows={len(baseline_overall)}")
print(" ", PATH_BY_SLICE, f"rows={len(baseline_by_slice)}")
if run_meta:
    print(" run_meta keys:", sorted(run_meta.keys())[:12], "...")
print(
    " k_cutoffs:",
    f"k_final(rank)={K_FINAL}",
    f"k_retrieval={K_RETRIEVAL}",
    f"k_personalization(diagnostics)={K_PERSONALIZATION}",
)

Loaded baseline tables:
  /home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_ranking_overall.csv rows=3
  /home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_ranking_by_slice.csv rows=6
 run_meta keys: ['active_cohort', 'archive_run', 'config_path', 'counts_by_slice', 'counts_by_support_bucket', 'coverage', 'k_final', 'k_personalization', 'k_retrieval', 'masking_policy_version', 'max_examples', 'methods_requested'] ...
 k_cutoffs: k_final(rank)=10 k_retrieval=100 k_personalization(diagnostics)=10


In [31]:
_display_sort_ranking_cols = [c for c in SORT_RANKING_OVERALL if c in required]
print(
    "\nBaseline ranking overall — primary: NDCG@K · secondary: MAP@K · tertiary: MRR "
    f"(k_final={K_FINAL}); tie-break: Hit@K → Precision@K → Recall@K"
)
display(baseline_overall_sub[required].sort_values(_display_sort_ranking_cols, ascending=False).reset_index(drop=True))

if PATH_TWO_TOWER_OVERALL.is_file():
    tt = pd.read_csv(PATH_TWO_TOWER_OVERALL)
    miss_tt = [c for c in required if c not in tt.columns]
    if miss_tt:
        raise ValueError(f"Two-tower overall CSV missing columns: {miss_tt}")
    tt_sub = tt[tt["method"].isin(CANDIDATE_METHODS)].copy()
    if tt_sub.empty:
        tt_sub = tt  # accept single-method file
    compare_overall = pd.concat([baseline_overall_sub[required], tt_sub[required]], ignore_index=True)
else:
    placeholder = pd.DataFrame(
        [{"method": m, **{metric: float("nan") for metric in METRIC_COLS}} for m in CANDIDATE_METHODS]
    )
    compare_overall = pd.concat([baseline_overall_sub[required], placeholder], ignore_index=True)

print(
    "\nComparison frame (baseline + candidate). "
    + (
        f"Loaded candidate from {PATH_TWO_TOWER_OVERALL}"
        if PATH_TWO_TOWER_OVERALL.is_file()
        else f"Candidate is NaN placeholder — add {PATH_TWO_TOWER_OVERALL.name} when ready."
    )
)
_cos = [c for c in SORT_RANKING_OVERALL if c in compare_overall.columns]
print(
    "Sort — primary: NDCG@K · secondary: MAP@K · tertiary: MRR "
    f"(k_final={K_FINAL}); tie-break: Hit@K → Precision@K → Recall@K"
)
display(compare_overall.sort_values(_cos, ascending=False).reset_index(drop=True))


Baseline ranking overall — primary: NDCG@K · secondary: MAP@K · tertiary: MRR (k_final=10); tie-break: Hit@K → Precision@K → Recall@K


,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR
0,popularity_train,0.15112,0.015184,0.146796,0.050594,0.073109,0.052221
1,multi_mean_train,0.08328,0.008376,0.076479,0.025349,0.037700,0.027358
2,raw,0.07168,0.007216,0.066606,0.024917,0.035020,0.026749



Comparison frame (baseline + candidate). Candidate is NaN placeholder — add eval_ranking_two_tower_overall.csv when ready.
Sort — primary: NDCG@K · secondary: MAP@K · tertiary: MRR (k_final=10); tie-break: Hit@K → Precision@K → Recall@K


,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR
0,popularity_train,0.15112,0.015184,0.146796,0.050594,0.073109,0.052221
1,multi_mean_train,0.08328,0.008376,0.076479,0.025349,0.037700,0.027358
2,raw,0.07168,0.007216,0.066606,0.024917,0.035020,0.026749
3,two_tower_a_raw_text,NaN,NaN,NaN,NaN,NaN,NaN
4,two_tower_b_raw_plus_mean_train,NaN,NaN,NaN,NaN,NaN,NaN
5,two_tower_c_raw_plus_behavior,NaN,NaN,NaN,NaN,NaN,NaN
6,two_tower_d_raw_plus_habit,NaN,NaN,NaN,NaN,NaN,NaN
7,two_tower_e_pref_structured_session,NaN,NaN,NaN,NaN,NaN,NaN


### Two-tower scoring (fill in later)

**Contract.** For each eval example, produce a **full-catalog** score vector aligned with `prepare_eval_inputs` / `run_retrieval_eval` (same `app_ids` order as `ContentRetriever`), then reuse the same ranking and metric definitions so `eval_ranking_*` / `eval_retrieval_*` tables stay comparable.

**Integration options.**

1. **Precomputed table** — Run a small script that writes `eval_ranking_two_tower_overall.csv` (and optionally by-slice) next to the baseline job outputs; this notebook will auto-pick it up (or use the legacy `eval_retrieval_two_tower_overall.csv` filename).
2. **In-notebook** — After loading `prepare_eval_inputs(...)` from `steam_review_ml.recommender.evaluation`, define `score_two_tower(ex) -> np.ndarray` and aggregate per-example metrics the same way as `run_retrieval_eval` (today `_per_example_metrics` is module-private; either extend the library with a named helper or keep the runner in a script).

**Artifacts** (TBD): e.g. `artifacts/recs/two_tower/` with vector dumps + `meta.json` mapping ids to `app_id`.

### Define two-tower artifact contract

This cell defines the expected artifact locations and metadata keys for in-notebook evaluation. We keep this explicit so failures are actionable (missing file/key tells you exactly what to export from training).

In [32]:
TWO_TOWER_DIR = REPO_ROOT / "artifacts" / "recs" / "two_tower"

# Optional save paths (no pre-existing vector files required).
PATH_TT_USER_VECS = TWO_TOWER_DIR / "user_vectors.npz"
PATH_TT_ITEM_VECS = TWO_TOWER_DIR / "item_vectors.npz"

PATH_COMPARE_OVERALL_OUT = BASELINE_EVAL_DIR / "eval_ranking_with_two_tower_overall.csv"
PATH_COMPARE_BY_SLICE_OUT = BASELINE_EVAL_DIR / "eval_ranking_with_two_tower_by_slice.csv"

print("Optional vector export dir:", TWO_TOWER_DIR)
print("Optional vector outputs:")
print(" -", PATH_TT_USER_VECS)
print(" -", PATH_TT_ITEM_VECS)
print("Comparison table outputs:")
print(" -", PATH_COMPARE_OVERALL_OUT)
print(" -", PATH_COMPARE_BY_SLICE_OUT)

Optional vector export dir: /home/ryanr/workspace/steam_recommendations/artifacts/recs/two_tower
Optional vector outputs:
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/two_tower/user_vectors.npz
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/two_tower/item_vectors.npz
Comparison table outputs:
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_ranking_with_two_tower_overall.csv
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_ranking_with_two_tower_by_slice.csv


### Local scoring + metric helpers

This cell adds notebook-local helpers that mirror the centralized eval job: **retrieval metrics** use `k_retrieval` (typically 100); **ranking metrics** (`MAP`, `NDCG`) use `k_final` (typically 10). Column names stay `Hit@K` etc.; the cutoff comes from config. Diagnostics (`CatalogCoverage@…`, `ILD@…`) use **top‑`k_personalization`** lists, aligned with job outputs that tag those columns `@10`.

In [33]:
from typing import Any

import numpy as np

from steam_review_ml.recommender.evaluation import (
    average_precision_at_k,
    hit_rate_at_k,
    mrr,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)


def _rank_rows(scores: np.ndarray) -> np.ndarray:
    if scores.ndim != 1:
        raise ValueError(f"Expected 1D score vector, got shape={scores.shape}")
    if np.isnan(scores).any():
        scores = np.nan_to_num(scores, nan=-np.inf)
    return np.argsort(-scores, kind="stable")


def _slice_name_from_n_targets(n_eval_targets: int) -> str:
    if n_eval_targets >= 2:
        return "slice_a_multi_target"
    if n_eval_targets == 1:
        return "slice_b_single_target"
    if n_eval_targets == 0:
        return "slice_c_zero_target"
    return "slice_other"


def _compute_metric_row(
    *,
    method: str,
    ranked_rows: np.ndarray,
    positives: set[int],
    app_ids: np.ndarray,
    k_final: int,
    n_eval_targets: int,
) -> dict[str, Any]:
    return {
        "method": method,
        "n_eval_targets": int(n_eval_targets),
        "slice_name": _slice_name_from_n_targets(int(n_eval_targets)),
        "Hit@K": hit_rate_at_k(ranked_rows, positives, k_final, app_ids),
        "Precision@K": precision_at_k(ranked_rows, positives, k_final, app_ids),
        "Recall@K": recall_at_k(ranked_rows, positives, k_final, app_ids),
        "MAP@K": average_precision_at_k(ranked_rows, positives, k_final, app_ids),
        "NDCG@K": ndcg_at_k(ranked_rows, positives, k_final, app_ids),
        "MRR": mrr(ranked_rows, positives, app_ids),
    }


def _compute_retrieval_metric_row(
    *,
    method: str,
    ranked_rows: np.ndarray,
    positives: set[int],
    app_ids: np.ndarray,
    k_retrieval: int,
    n_eval_targets: int,
) -> dict[str, Any]:
    return {
        "method": method,
        "n_eval_targets": int(n_eval_targets),
        "slice_name": _slice_name_from_n_targets(int(n_eval_targets)),
        "Hit@K": hit_rate_at_k(ranked_rows, positives, k_retrieval, app_ids),
        "Precision@K": precision_at_k(ranked_rows, positives, k_retrieval, app_ids),
        "Recall@K": recall_at_k(ranked_rows, positives, k_retrieval, app_ids),
    }

## Temporary check: normalized playtime fields in processed train split

This is a temporary preflight check to confirm `_norm_*` playtime features exist before loading eval examples.

In [34]:
CHECK_FOR_NORM_COLS = False

if CHECK_FOR_NORM_COLS:
    TRAIN_NORM_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_train_norm.parquet"
    VAL_NORM_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_val_norm.parquet"
    CHECK_NORM_COLS = [
        "_norm_author__playtime_at_review",
        "_norm_author__playtime_last_two_weeks",
        "_norm_author__num_games_owned",
        "_norm_author__num_reviews",
        "_norm_review_word_count",
    ]

    if not TRAIN_NORM_PATH.is_file():
        raise FileNotFoundError(f"Missing processed train parquet: {TRAIN_NORM_PATH}")
    if not VAL_NORM_PATH.is_file():
        raise FileNotFoundError(f"Missing processed val parquet: {VAL_NORM_PATH}")

    df_tmp = pd.read_parquet(TRAIN_NORM_PATH)
    df_tmp_val = pd.read_parquet(VAL_NORM_PATH)
    def check_norm_cols(df, df_name, cols, file_path):
        present = [c for c in cols if c in df.columns]
        missing = [c for c in cols if c not in df.columns]
        print(f"Processed {df_name} file:", file_path)
        print("Present normalized columns:", present)
        print("Missing normalized columns:", missing)
        if present:
            non_null_counts = {c: int(df[c].notna().sum()) for c in present}
            print("Non-null counts:", non_null_counts)
            print(df[present].head(3))

    check_norm_cols(df_tmp, "train", CHECK_NORM_COLS, TRAIN_NORM_PATH)
    check_norm_cols(df_tmp_val, "val", CHECK_NORM_COLS, VAL_NORM_PATH)
else:
    print("Skipping normalized column check.")


Skipping normalized column check.


## Load eval cohort and two-tower artifacts

This cell builds the same eval examples used by the baseline job (`prepare_eval_inputs`) and loads two-tower vectors + id maps. It fails early on shape/id mismatches to avoid silent metric drift.

In [35]:
from collections import Counter, defaultdict

from steam_review_ml.constants import PROJECT_RANDOM_SEED
from steam_review_ml.recommender.evaluation import prepare_eval_inputs
from steam_review_ml.recommender.math_utils import l2_normalize
from steam_review_ml.recommender.retrieve import ContentRetriever

# Clean conceptual split:
# - Behavior weighting uses only game-tied playtime-at-review.
# - Context scalars are user/activity features kept separate.
PLAYTIME_CANDIDATE_KEYS = ("_norm_author__playtime_at_review",)
CONTEXT_SCALAR_KEYS = (
    "_norm_author__playtime_last_two_weeks",
    "_norm_author__num_games_owned",
    "_norm_author__num_reviews",
)

def _validate_eval_config(cfg: dict) -> None:
    """
    Ensure all required keys are present in the evaluation config.
    This function guards against missing or malformed configs at runtime.
    """
    required_keys = (
        "split",
        "active_cohort",
        "max_examples",
        "support_app_filter_mode",
        "min_review_chars",
        "max_train_rows_per_user",
    )
    missing = [k for k in required_keys if k not in cfg]
    if missing:
        raise ValueError(f"Missing required eval config keys: {missing}")

def _playtime_key_coverage(rows: list[dict]) -> dict[str, int]:
    """
    Count how often each candidate playtime key appears with non-null values.
    """
    counts = Counter()
    for row in rows:
        for key in PLAYTIME_CANDIDATE_KEYS:
            if key in row and row[key] is not None:
                counts[key] += 1
    return {k: int(counts[k]) for k in PLAYTIME_CANDIDATE_KEYS}


def _coverage(rows: list[dict], keys: tuple[str, ...]) -> dict[str, int]:
    """Generic non-null key coverage counter for row dicts."""
    counts = Counter()
    for row in rows:
        for key in keys:
            if key in row and row[key] is not None:
                counts[key] += 1
    return {k: int(counts[k]) for k in keys}


def _validate_playtime_fields(rows: list[dict]) -> dict[str, int]:
    """
    Diagnostic-only validation for playtime fields.
    Prints coverage and available keys; does not raise.
    """
    if not rows:
        print("Warning: no train review rows found; behavior weighting will use fallback.")
        return {k: 0 for k in PLAYTIME_CANDIDATE_KEYS}

    coverage = _playtime_key_coverage(rows)
    sample_keys = sorted(rows[0].keys()) if rows else []

    if not any(coverage.values()):
        print(
            "Warning: no usable playtime fields found in train_review_rows. "
            f"Checked keys: {list(PLAYTIME_CANDIDATE_KEYS)}"
        )
        print("Available keys in train_review_rows (sample):", sample_keys)
        print("Using unweighted behavior fallback where needed.")
    else:
        print("Playtime key coverage:", coverage)

    return coverage

def _extract_playtime_weight(row: dict) -> float:
    """
    Extract pre-normalized playtime weight from a review row.
    Uses the maximum positive value across candidate normalized fields.
    """
    vals = []
    for key in PLAYTIME_CANDIDATE_KEYS:
        if key in row and row[key] is not None:
            vals.append(max(0.0, float(row[key])))
    if not vals:
        return 0.0
    return float(max(vals))

def _build_u_reviews(ex: dict, *, retriever) -> np.ndarray:
    """Text-history embedding, independent of behavior weighting."""
    support_texts = [
        str(r.get("text", "")).strip()
        for r in ex.get("train_review_rows", [])
        if str(r.get("text", "")).strip()
    ]
    if support_texts:
        vecs = np.stack([retriever.embed_text(t) for t in support_texts], axis=0).astype(np.float32)
        return l2_normalize(vecs.mean(axis=0))
    return retriever.embed_text(str(ex.get("query_text", "")))

def _build_u_behavior_weighted(
    ex: dict,
    *,
    X: np.ndarray,
    app_to_row: dict[int, int],
    fallback: np.ndarray,
) -> tuple[np.ndarray, bool]:
    """Per-game behavior embedding weighted by playtime-at-review only."""
    rows = ex.get("train_review_rows", [])
    weighted_vecs = []
    weights = []
    for r in rows:
        app_id = int(r.get("app_id", -1))
        row_idx = app_to_row.get(app_id)
        if row_idx is None:
            continue  # No embedding available for this app.
        w = _extract_playtime_weight(r)
        if w <= 0.0:
            continue
        weighted_vecs.append(X[row_idx].astype(np.float32))
        weights.append(w)

    if weighted_vecs:
        mat = np.stack(weighted_vecs, axis=0)
        w_arr = np.asarray(weights, dtype=np.float32)
        vec = (mat * w_arr[:, None]).sum(axis=0) / np.maximum(w_arr.sum(), 1e-12)
        return l2_normalize(vec), True

    # Fallback retained for robustness; explicitly tracked in diagnostics.
    support_app_ids = sorted({int(r.get("app_id")) for r in rows if int(r.get("app_id", -1)) in app_to_row})
    if support_app_ids:
        mat = np.stack([X[app_to_row[a]] for a in support_app_ids], axis=0).astype(np.float32)
        return l2_normalize(mat.mean(axis=0)), False

    return fallback, False

def _build_user_context_scalars(ex: dict) -> dict[str, float]:
    """Current/user context scalars are kept separate from behavior weighting."""
    rows = ex.get("train_review_rows", [])
    out: dict[str, float] = {}
    for key in CONTEXT_SCALAR_KEYS:
        vals = [float(r[key]) for r in rows if key in r and r[key] is not None]
        out[key] = float(np.mean(vals)) if vals else 0.0
    return out


def _load_examples_from_cache(path: Path) -> list[dict]:
    """Load eval examples cache written by recs_job_build_eval_examples.py."""
    df = pd.read_parquet(path)
    required_cols = {
        "user_id",
        "query_app_id",
        "query_text",
        "query_ts",
        "n_eval_targets",
        "cohort",
        "eval_pos_cohort",
        "validation_positive_app_ids_json",
        "train_review_rows_json",
    }
    missing = sorted(c for c in required_cols if c not in df.columns)
    if missing:
        raise ValueError(f"Cached examples missing required columns: {missing}")

    examples: list[dict] = []
    for row in df.itertuples(index=False):
        rec = row._asdict()
        examples.append(
            {
                "user_id": str(rec["user_id"]),
                "query_app_id": int(rec["query_app_id"]),
                "query_text": str(rec["query_text"]),
                "query_ts": float(rec["query_ts"]),
                "validation_positive_app_ids": set(
                    int(a) for a in json.loads(rec["validation_positive_app_ids_json"])
                ),
                "n_eval_targets": int(rec["n_eval_targets"]),
                "train_review_rows": json.loads(rec["train_review_rows_json"]),
                "cohort": str(rec["cohort"]),
                "eval_pos_cohort": str(rec["eval_pos_cohort"]),
            }
        )
    return examples


In [36]:
from steam_review_ml.recommender import build_embedding_input, extract_preferences

# 1) Validate config and choose examples source (cache-first, fallback to prepare_eval_inputs)
print("Loading config and validating required keys...")
cfg = load_config(str(EVAL_JOB_CONFIG))
_validate_eval_config(cfg)
print("Config loaded and validated.")

USE_EXAMPLES_CACHE = True
EVAL_EXAMPLES_CACHE = (
    REPO_ROOT
    / "artifacts"
    / "recs"
    / "eval_cache"
    / "val_dev_12k_v1"
    / "eval_examples.parquet"
)

print("Loading shared retriever/item embeddings...")
retriever = ContentRetriever(repo_root=REPO_ROOT)
catalog_app_ids = np.asarray(retriever.app_ids)
X = np.asarray(retriever.embedding_matrix)
app_to_row = {int(a): i for i, a in enumerate(catalog_app_ids.tolist())}

if X.ndim != 2:
    raise ValueError(f"Expected 2D embedding matrix, got shape={X.shape}")
if len(catalog_app_ids) != X.shape[0]:
    raise ValueError("app_ids length must match embedding matrix rows")

cache_loaded = False
if USE_EXAMPLES_CACHE and EVAL_EXAMPLES_CACHE.is_file():
    print(f"Loading eval examples from cache: {EVAL_EXAMPLES_CACHE}")
    examples = _load_examples_from_cache(EVAL_EXAMPLES_CACHE)
    cache_loaded = True
else:
    if USE_EXAMPLES_CACHE:
        print(f"Cache not found at {EVAL_EXAMPLES_CACHE}; falling back to prepare_eval_inputs().")
    else:
        print("USE_EXAMPLES_CACHE=False; using prepare_eval_inputs().")

    prepared = prepare_eval_inputs(
        repo_root=REPO_ROOT,
        split=str(cfg.get("split", "val")),
        active_cohort=str(cfg.get("active_cohort", "all")),
        max_examples=int(cfg.get("max_examples", 12_500)),
        support_app_filter_mode=str(cfg.get("support_app_filter_mode", "strict")),
        cohort_sizing={
            tuple(k.split("|", 1)): float(v)
            for k, v in dict(cfg.get("cohort_sizing", {})).items()
        },
        min_review_chars=int(cfg.get("min_review_chars", 30)),
        max_train_rows_per_user=int(cfg.get("max_train_rows_per_user", 5)),
        random_seed=int(cfg.get("random_seed", PROJECT_RANDOM_SEED)),
        artifact_dir=REPO_ROOT / str(cfg.get("artifact_dir", "artifacts/recs")),
        verbose=False,
    )
    examples = prepared.examples

if not examples:
    raise RuntimeError("No evaluation examples available from cache or prepare_eval_inputs")

from types import SimpleNamespace

inputs = SimpleNamespace(
    examples=examples,
    retriever=retriever,
    app_ids=catalog_app_ids,
    embedding_matrix=X,
    app_to_row=app_to_row,
    source="cache" if cache_loaded else "prepare_eval_inputs",
)

print(f"Loaded {len(inputs.examples)} eval examples from {inputs.source}.")
print(f"Catalog embeddings: {X.shape[0]} items x {X.shape[1]} dims.")

# 2) Prepare catalog/item matrix
catalog_item_matrix = X.astype(np.float32, copy=False)
print("Catalog item matrix ready. Shape:", catalog_item_matrix.shape)

# 3) Validate behavior/context field availability
print("Validating behavior/context fields on train review rows...")
all_train_rows = [r for ex in inputs.examples for r in ex.get("train_review_rows", [])]
_ = _validate_playtime_fields(all_train_rows)
context_coverage = _coverage(all_train_rows, CONTEXT_SCALAR_KEYS)
print("Context scalar key coverage:", context_coverage)
positive_weight_rows = sum(1 for r in all_train_rows if _extract_playtime_weight(r) > 0.0)
print(f"Rows with positive behavior weight ({PLAYTIME_CANDIDATE_KEYS[0]}): {positive_weight_rows}/{len(all_train_rows)}")

# 4) Build per-example vectors for A/B/C/D candidates
ex_vectors_by_idx: dict[int, dict[str, np.ndarray]] = {}
vector_rows = []
n_weighted_examples = 0
n_fallback_examples = 0

for ex_idx, ex in enumerate(inputs.examples):
    query_text_raw = str(ex.get("query_text", ""))
    q_session = inputs.retriever.embed_text(query_text_raw)
    q_pref_structured = inputs.retriever.embed_text(
        build_embedding_input(extract_preferences(query_text_raw), query_text_raw)
    )
    u_reviews = _build_u_reviews(ex, retriever=inputs.retriever)
    u_behavior, used_weighted = _build_u_behavior_weighted(ex, X=X, app_to_row=app_to_row, fallback=u_reviews)
    if used_weighted:
        n_weighted_examples += 1
    else:
        n_fallback_examples += 1

    u_habit = l2_normalize(0.5 * u_behavior + 0.5 * u_reviews)
    two_tower_a_raw_text = q_session
    two_tower_b_raw_plus_mean_train = l2_normalize(q_session + u_reviews)
    two_tower_c_raw_plus_behavior = l2_normalize(q_session + u_behavior)
    two_tower_d_raw_plus_habit = l2_normalize(q_session + u_habit)
    two_tower_e_pref_structured_session = q_pref_structured
    context_scalars = _build_user_context_scalars(ex)

    ex_vectors_by_idx[ex_idx] = {
        "two_tower_a_raw_text": two_tower_a_raw_text.astype(np.float32),
        "two_tower_b_raw_plus_mean_train": two_tower_b_raw_plus_mean_train.astype(np.float32),
        "two_tower_c_raw_plus_behavior": two_tower_c_raw_plus_behavior.astype(np.float32),
        "two_tower_d_raw_plus_habit": two_tower_d_raw_plus_habit.astype(np.float32),
        "two_tower_e_pref_structured_session": two_tower_e_pref_structured_session.astype(np.float32),
    }

    user_id = str(ex.get("user_id"))

    vector_rows.append(
        {
            "ex_idx": ex_idx,
            "user_id": user_id,
            "query_app_id": int(ex.get("query_app_id")),
            "q_session": q_session,
            "u_reviews": u_reviews,
            "u_behavior": u_behavior,
            "u_habit": u_habit,
            **context_scalars,
        }
    )

print("Generated in-notebook candidate vectors")
print("  examples:", len(inputs.examples))
print("  examples with vectors:", len(ex_vectors_by_idx))
print("  weighted behavior examples:", n_weighted_examples)
print("  fallback behavior examples:", n_fallback_examples)
print("  catalog size:", len(catalog_app_ids))
print("  embedding dimension:", catalog_item_matrix.shape[1])

Loading config and validating required keys...
Config loaded and validated.
Loading shared retriever/item embeddings...
Loading eval examples from cache: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
Loaded 12500 eval examples from cache.
Catalog embeddings: 315 items x 512 dims.
Catalog item matrix ready. Shape: (315, 512)
Validating behavior/context fields on train review rows...
Playtime key coverage: {'_norm_author__playtime_at_review': 24032}
Context scalar key coverage: {'_norm_author__playtime_last_two_weeks': 24032, '_norm_author__num_games_owned': 24032, '_norm_author__num_reviews': 24032}
Rows with positive behavior weight (_norm_author__playtime_at_review): 23984/24032
Generated in-notebook candidate vectors
  examples: 12500
  examples with vectors: 12500
  weighted behavior examples: 9370
  fallback behavior examples: 3130
  catalog size: 315
  embedding dimension: 512


In [37]:
pd.DataFrame(all_train_rows).head()

,app_id,text,ts,_norm_author__playtime_at_review,_norm_author__playtime_last_two_weeks,_norm_author__num_games_owned,_norm_author__num_reviews,_norm_review_word_count
0,359550,Facepalm as your teammates run into marked ene...,1.473255e+09,7.486613,0.0,6.906755,2.995732,3.044522
1,571740,> Download Game\n> Get some rage maps off the ...,1.542887e+09,6.285998,0.0,6.907755,2.995732,2.995732
2,899440,"As a long term Monster Hunter fan, this itches...",1.549898e+09,7.702556,0.0,5.579730,2.484907,3.988984
3,945360,"Claimed someone was a murder, followed her aro...",1.600532e+09,4.369448,0.0,5.579730,2.484907,3.465736
4,582010,As a Monster Hunter fan from the PSP / PS2 era...,1.534964e+09,8.237479,0.0,5.579730,2.484907,4.043051


## Score examples and aggregate contract tables

This cell computes per-example scores via user-vector dot product against catalog item vectors, then produces:

- **Ranking (k_final):** `two_tower_overall`, `two_tower_by_slice` — same columns as `eval_ranking_*.csv` (including Hit/Precision/Recall **at ranking depth** alongside MAP/NDCG/MRR).
- **Retrieval (k_retrieval):** `two_tower_retrieval_overall`, `two_tower_retrieval_by_slice` — Hit/Precision/Recall only, aligned with `eval_retrieval_*.csv`.
- **Diagnostics:** `@K` suffix on guardrail columns uses **k_personalization** (typically 10), matching job output naming (`CatalogCoverage@10`, …).

### How `l2_normalize(q_session + u_habit)` works

- `q_session` and `u_habit` are two embedding vectors in the same space.
- `q_session + u_habit` blends current intent (`q_session`) with historical preference (`u_habit`).
- `l2_normalize(...)` rescales the blended vector to unit length so score magnitude does not depend on vector norm.
- Actual retrieval scoring is `scores = catalog_item_matrix @ user_query_vector`, then ranking uses `_rank_rows(scores)`.

In [38]:
inputs.examples[0]

{'user_id': '76561198001296435',
 'query_app_id': 812140,
 'query_text': "> Got it on Sale\n> Started it up, played Kassandra 'coz better voice acting imo\n> Played a bit, unlock boat\n> Destroyed a bunch of ships\n> Rammed a bunch of ships\n> Boarded a bunch of ships\n> Wait there's more story after you unlock the boat?\n> Meh, RAMMING SPEED!!!",
 'query_ts': 1568427802.0,
 'validation_positive_app_ids': {262060},
 'n_eval_targets': 1,
 'train_review_rows': [{'app_id': 359550,
   'text': 'Facepalm as your teammates run into marked enemies before being outnumbered and killed yourself, followed by a complementary teabagging. 11/10',
   'ts': 1473255297.0,
   '_norm_author__playtime_at_review': 7.486613313139955,
   '_norm_author__playtime_last_two_weeks': 0.0,
   '_norm_author__num_games_owned': 6.906754778648554,
   '_norm_author__num_reviews': 2.995732273553991,
   '_norm_review_word_count': 3.044522437723423},
  {'app_id': 571740,
   'text': '> Download Game\n> Get some rage maps off

In [39]:
rows = []
rows_retr = []
top_rows_by_method: dict[str, list[np.ndarray]] = {m: [] for m in CANDIDATE_METHODS}
skipped_no_example_vector = 0

for ex_idx, ex in enumerate(inputs.examples):
    validation_positive_app_ids = set(int(a) for a in ex.get("validation_positive_app_ids", set()))
    if not validation_positive_app_ids:
        continue

    q_app = int(ex.get("query_app_id"))
    n_eval_targets = int(ex.get("n_eval_targets", len(validation_positive_app_ids)))

    candidate_vectors = ex_vectors_by_idx.get(ex_idx)
    if candidate_vectors is None:
        skipped_no_example_vector += 1
        continue

    q_idx = inputs.app_to_row.get(q_app)
    for method_name in CANDIDATE_METHODS:
        u = candidate_vectors[method_name]
        scores = catalog_item_matrix @ u.astype(np.float32)

        # Keep query app masked to match baseline retrieval evaluation behavior.
        if q_idx is not None:
            scores[q_idx] = -np.inf

        ranked_rows = _rank_rows(scores)
        top_rows_by_method[method_name].append(ranked_rows[:K_PERSONALIZATION])

        rows.append(
            _compute_metric_row(
                method=method_name,
                ranked_rows=ranked_rows,
                positives=validation_positive_app_ids,
                app_ids=catalog_app_ids,
                k_final=K_FINAL,
                n_eval_targets=n_eval_targets,
            )
        )
        rows_retr.append(
            _compute_retrieval_metric_row(
                method=method_name,
                ranked_rows=ranked_rows,
                positives=validation_positive_app_ids,
                app_ids=catalog_app_ids,
                k_retrieval=K_RETRIEVAL,
                n_eval_targets=n_eval_targets,
            )
        )

if not rows:
    raise RuntimeError("No two-tower rows scored. Check user id alignment and vector artifacts.")

df_tt_ex = pd.DataFrame(rows)
df_tt_ex_retrieval = pd.DataFrame(rows_retr)
two_tower_retrieval_overall = (
    df_tt_ex_retrieval.groupby("method", observed=True)[RETRIEVAL_METRIC_COLS]
    .mean()
    .reset_index()
)
two_tower_retrieval_by_slice = (
    df_tt_ex_retrieval.groupby(["slice_name", "method"], observed=True)[RETRIEVAL_METRIC_COLS]
    .mean()
    .reset_index()
)
two_tower_retrieval_by_slice = _sort_retrieval_by_slice_for_display(two_tower_retrieval_by_slice)
two_tower_overall = (
    df_tt_ex.groupby("method", observed=True)[METRIC_COLS]
    .mean()
    .reset_index()
)

app_counts_proxy = Counter(int(r.get("app_id")) for r in all_train_rows if r.get("app_id") is not None)
pop_proxy = np.asarray([float(app_counts_proxy.get(int(a), 0.0)) for a in catalog_app_ids], dtype=np.float32)
pop_proxy = np.maximum(pop_proxy, 1e-6)
pop_share = (pop_proxy + 1.0) / float(pop_proxy.sum() + len(pop_proxy))
item_novelty = -np.log2(pop_share)
pop_top_rows = np.argsort(-pop_proxy)[:K_PERSONALIZATION]
pop_top_apps = set(int(catalog_app_ids[i]) for i in pop_top_rows)

diag_rows = []
for method_name, top_rows_list in top_rows_by_method.items():
    if not top_rows_list:
        continue
    seen_items: set[int] = set()
    ild_vals: list[float] = []
    novelty_vals: list[float] = []
    gap_vals: list[float] = []
    for top_rows in top_rows_list:
        top_apps = set(int(catalog_app_ids[i]) for i in top_rows)
        seen_items.update(top_apps)

        emb = catalog_item_matrix[top_rows]
        if len(top_rows) <= 1:
            ild_vals.append(0.0)
        else:
            sim = emb @ emb.T
            tri = np.triu_indices(len(top_rows), k=1)
            ild_vals.append(float(np.mean(1.0 - sim[tri])) if len(tri[0]) else 0.0)

        novelty_vals.append(float(np.mean(item_novelty[top_rows])))
        union = top_apps | pop_top_apps
        jacc = float(len(top_apps & pop_top_apps) / len(union)) if union else 1.0
        gap_vals.append(1.0 - jacc)

    diag_rows.append(
        {
            "method": method_name,
            f"CatalogCoverage@{K_PERSONALIZATION}": float(len(seen_items) / len(catalog_app_ids)),
            f"ILD@{K_PERSONALIZATION}": float(np.mean(ild_vals)),
            f"Novelty@{K_PERSONALIZATION}": float(np.mean(novelty_vals)),
            f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}": float(np.mean(gap_vals)),
        }
    )

diag_df = pd.DataFrame(diag_rows)
if not diag_df.empty:
    two_tower_overall = two_tower_overall.merge(diag_df, on="method", how="left")
    two_tower_retrieval_overall = two_tower_retrieval_overall.merge(diag_df, on="method", how="left")

two_tower_by_slice = (
    df_tt_ex.groupby(["slice_name", "method"], observed=True)[METRIC_COLS]
    .mean()
    .reset_index()
)
if not diag_df.empty:
    two_tower_by_slice = two_tower_by_slice.merge(diag_df, on="method", how="left")
two_tower_by_slice = _sort_ranking_by_slice_for_display(two_tower_by_slice)

coverage_summary = {
    "n_total": int(len(inputs.examples)),
    "n_multi_pos": int(sum(1 for ex in inputs.examples if int(ex.get("n_eval_targets", 0)) >= 2)),
    "n_single_pos": int(sum(1 for ex in inputs.examples if int(ex.get("n_eval_targets", 0)) == 1)),
    "n_zero_pos": int(sum(1 for ex in inputs.examples if int(ex.get("n_eval_targets", 0)) == 0)),
}
coverage_summary["coverage_multi_pos"] = (
    float(coverage_summary["n_multi_pos"] / coverage_summary["n_total"])
    if coverage_summary["n_total"]
    else float("nan")
)

retrieval_policy_slice_a = (
    two_tower_retrieval_by_slice[two_tower_retrieval_by_slice["slice_name"] == "slice_a_multi_target"]
    .sort_values(["Recall@K", "Precision@K", "Hit@K"], ascending=False)
    .reset_index(drop=True)
)
retrieval_policy_slice_b = (
    two_tower_retrieval_by_slice[two_tower_retrieval_by_slice["slice_name"] == "slice_b_single_target"]
    .sort_values(["Hit@K", "Precision@K", "Recall@K"], ascending=False)
    .reset_index(drop=True)
)

ranking_policy_slice_a = (
    two_tower_by_slice[two_tower_by_slice["slice_name"] == "slice_a_multi_target"]
    .sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False)
    .reset_index(drop=True)
)
ranking_policy_slice_b = (
    two_tower_by_slice[two_tower_by_slice["slice_name"] == "slice_b_single_target"]
    .sort_values(["Hit@K", "MRR", "NDCG@K", "MAP@K"], ascending=False)
    .reset_index(drop=True)
)

print("Two-tower scoring complete")
print(" methods:", CANDIDATE_METHODS)
print(
    " k_cutoffs:",
    f"k_final(rank)={K_FINAL}",
    f"k_retrieval={K_RETRIEVAL}",
    f"k_personalization(diagnostics)={K_PERSONALIZATION}",
)
print(" scored rows:", len(df_tt_ex))
print(" skipped (missing example vector):", skipped_no_example_vector)
print(" coverage summary:", coverage_summary)

print("\nTwo-tower ranking overall — primary: NDCG@K · secondary: MAP@K · tertiary: MRR " f"(k_final={K_FINAL}); tie-breakers: Hit → Precision → Recall")
display(two_tower_overall.sort_values(SORT_RANKING_OVERALL, ascending=False).reset_index(drop=True))

print(
    "\nTwo-tower ranking by slice — slice A multi-target: primary NDCG@K · secondary MAP@K · tertiary MRR; "
    "slice B single-target: primary Hit@K · secondary MRR; then tie-break with remaining ranking columns"
)
display(two_tower_by_slice)

print(
    f"\nTwo-tower retrieval overall @ k_retrieval={K_RETRIEVAL} — primary: Recall@K · secondary: Precision@K · tertiary: Hit@K"
)
display(two_tower_retrieval_overall.sort_values(SORT_RETRIEVAL_OVERALL, ascending=False).reset_index(drop=True))

print(
    "\nTwo-tower retrieval by slice — slice A: primary Recall@K · secondary Precision@K · tertiary Hit@K; "
    "slice B: primary Hit@K · secondary Precision@K · tertiary Recall@K"
)
display(two_tower_retrieval_by_slice)

print("\nRetrieval (candidates) — slice A — primary: Recall@K · secondary: Precision@K · tertiary: Hit@K")
display(retrieval_policy_slice_a)
print("\nRetrieval (candidates) — slice B — primary: Hit@K · secondary: Precision@K · tertiary: Recall@K")
display(retrieval_policy_slice_b)
print("\nRanking (candidates) — slice A — primary: NDCG@K · secondary: MAP@K · tertiary: MRR")
display(ranking_policy_slice_a)
print("\nRanking (candidates) — slice B — primary: Hit@K · secondary: MRR · tertiary: NDCG@K (then MAP)")
display(ranking_policy_slice_b)

Two-tower scoring complete
 methods: ['two_tower_a_raw_text', 'two_tower_b_raw_plus_mean_train', 'two_tower_c_raw_plus_behavior', 'two_tower_d_raw_plus_habit', 'two_tower_e_pref_structured_session']
 k_cutoffs: k_final(rank)=10 k_retrieval=100 k_personalization(diagnostics)=10
 scored rows: 62500
 skipped (missing example vector): 0
 coverage summary: {'n_total': 12500, 'n_multi_pos': 725, 'n_single_pos': 11775, 'n_zero_pos': 0, 'coverage_multi_pos': 0.058}

Two-tower ranking overall — primary: NDCG@K · secondary: MAP@K · tertiary: MRR (k_final=10); tie-breakers: Hit → Precision → Recall


,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,two_tower_d_raw_plus_habit,0.08376,0.008424,0.077867,0.026579,0.038979,0.043440,1.000000,0.138554,9.661580,0.976607
1,two_tower_c_raw_plus_behavior,0.08536,0.008640,0.079430,0.025877,0.038836,0.042861,1.000000,0.134907,9.574771,0.973837
2,two_tower_b_raw_plus_mean_train,0.08152,0.008192,0.075468,0.025926,0.037861,0.042131,1.000000,0.147905,9.719181,0.979075
3,two_tower_a_raw_text,0.07168,0.007216,0.066606,0.024917,0.035020,0.040381,1.000000,0.160723,9.734209,0.980731
4,two_tower_e_pref_structured_session,0.04664,0.004696,0.042510,0.015930,0.022508,0.028657,0.996825,0.172494,10.428898,0.994008



Two-tower ranking by slice — slice A multi-target: primary NDCG@K · secondary MAP@K · tertiary MRR; slice B single-target: primary Hit@K · secondary MRR; then tie-break with remaining ranking columns


,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_a_multi_target,two_tower_c_raw_plus_behavior,0.172414,0.019034,0.070175,0.025696,0.046757,0.082511,1.000000,0.134907,9.574771,0.973837
1,slice_a_multi_target,two_tower_b_raw_plus_mean_train,0.171034,0.017793,0.066697,0.022662,0.042981,0.076630,1.000000,0.147905,9.719181,0.979075
2,slice_a_multi_target,two_tower_d_raw_plus_habit,0.162759,0.017103,0.061161,0.023563,0.042571,0.080748,1.000000,0.138554,9.661580,0.976607
3,slice_a_multi_target,two_tower_a_raw_text,0.143448,0.015172,0.055964,0.021677,0.038812,0.073125,1.000000,0.160723,9.734209,0.980731
4,slice_a_multi_target,two_tower_e_pref_structured_session,0.118621,0.012414,0.047418,0.018478,0.032639,0.060171,0.996825,0.172494,10.428898,0.994008
5,slice_b_single_target,two_tower_c_raw_plus_behavior,0.080000,0.008000,0.080000,0.025888,0.038348,0.040419,1.000000,0.134907,9.574771,0.973837
6,slice_b_single_target,two_tower_d_raw_plus_habit,0.078896,0.007890,0.078896,0.026764,0.038758,0.041143,1.000000,0.138554,9.661580,0.976607
7,slice_b_single_target,two_tower_b_raw_plus_mean_train,0.076008,0.007601,0.076008,0.026127,0.037546,0.040006,1.000000,0.147905,9.719181,0.979075
8,slice_b_single_target,two_tower_a_raw_text,0.067261,0.006726,0.067261,0.025117,0.034786,0.038365,1.000000,0.160723,9.734209,0.980731
9,slice_b_single_target,two_tower_e_pref_structured_session,0.042208,0.004221,0.042208,0.015773,0.021884,0.026717,0.996825,0.172494,10.428898,0.994008



Two-tower retrieval overall @ k_retrieval=100 — primary: Recall@K · secondary: Precision@K · tertiary: Hit@K


...

,slice_name,method,Hit@K,Precision@K,Recall@K
0,slice_b_single_target,two_tower_c_raw_plus_behavior,0.489597,0.004896,0.489597
1,slice_b_single_target,two_tower_d_raw_plus_habit,0.482208,0.004822,0.482208
2,slice_b_single_target,two_tower_b_raw_plus_mean_train,0.468535,0.004685,0.468535
3,slice_b_single_target,two_tower_a_raw_text,0.434820,0.004348,0.434820
4,slice_b_single_target,two_tower_e_pref_structured_session,0.326200,0.003262,0.326200



Ranking (candidates) — slice A — primary: NDCG@K · secondary: MAP@K · tertiary: MRR


,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_a_multi_target,two_tower_c_raw_plus_behavior,0.172414,0.019034,0.070175,0.025696,0.046757,0.082511,1.000000,0.134907,9.574771,0.973837
1,slice_a_multi_target,two_tower_b_raw_plus_mean_train,0.171034,0.017793,0.066697,0.022662,0.042981,0.076630,1.000000,0.147905,9.719181,0.979075
2,slice_a_multi_target,two_tower_d_raw_plus_habit,0.162759,0.017103,0.061161,0.023563,0.042571,0.080748,1.000000,0.138554,9.661580,0.976607
3,slice_a_multi_target,two_tower_a_raw_text,0.143448,0.015172,0.055964,0.021677,0.038812,0.073125,1.000000,0.160723,9.734209,0.980731
4,slice_a_multi_target,two_tower_e_pref_structured_session,0.118621,0.012414,0.047418,0.018478,0.032639,0.060171,0.996825,0.172494,10.428898,0.994008



Ranking (candidates) — slice B — primary: Hit@K · secondary: MRR · tertiary: NDCG@K (then MAP)


,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_b_single_target,two_tower_c_raw_plus_behavior,0.080000,0.008000,0.080000,0.025888,0.038348,0.040419,1.000000,0.134907,9.574771,0.973837
1,slice_b_single_target,two_tower_d_raw_plus_habit,0.078896,0.007890,0.078896,0.026764,0.038758,0.041143,1.000000,0.138554,9.661580,0.976607
2,slice_b_single_target,two_tower_b_raw_plus_mean_train,0.076008,0.007601,0.076008,0.026127,0.037546,0.040006,1.000000,0.147905,9.719181,0.979075
3,slice_b_single_target,two_tower_a_raw_text,0.067261,0.006726,0.067261,0.025117,0.034786,0.038365,1.000000,0.160723,9.734209,0.980731
4,slice_b_single_target,two_tower_e_pref_structured_session,0.042208,0.004221,0.042208,0.015773,0.021884,0.026717,0.996825,0.172494,10.428898,0.994008


## Merge with baseline and optionally write outputs

This cell concatenates baseline and candidate tables and can save them as new files without mutating the original baseline artifacts.

In [40]:
WRITE_OUTPUTS = False

baseline_by_slice_sub = baseline_by_slice[baseline_by_slice["method"].isin(BASELINE_METHODS)].copy()

baseline_retr_overall_sub = pd.DataFrame(columns=["method", *RETRIEVAL_METRIC_COLS])
baseline_retr_by_slice_sub = pd.DataFrame(columns=["slice_name", "method", *RETRIEVAL_METRIC_COLS])
if PATH_RETRIEVAL_OVERALL.is_file():
    baseline_retr_overall = pd.read_csv(PATH_RETRIEVAL_OVERALL)
    miss_r = [c for c in RETRIEVAL_METRIC_COLS if c not in baseline_retr_overall.columns]
    if miss_r:
        raise ValueError(f"{PATH_RETRIEVAL_OVERALL} missing columns: {miss_r}")
    baseline_retr_overall_sub = baseline_retr_overall[baseline_retr_overall["method"].isin(BASELINE_METHODS)][
        ["method", *RETRIEVAL_METRIC_COLS]
    ].copy()
if PATH_RETRIEVAL_BY_SLICE.is_file():
    baseline_retr_by_slice = pd.read_csv(PATH_RETRIEVAL_BY_SLICE)
    miss_s = [c for c in RETRIEVAL_METRIC_COLS if c not in baseline_retr_by_slice.columns]
    if miss_s:
        raise ValueError(f"{PATH_RETRIEVAL_BY_SLICE} missing columns: {miss_s}")
    baseline_retr_by_slice_sub = baseline_retr_by_slice[baseline_retr_by_slice["method"].isin(BASELINE_METHODS)][
        ["slice_name", "method", *RETRIEVAL_METRIC_COLS]
    ].copy()
if not PATH_RETRIEVAL_OVERALL.is_file() or not PATH_RETRIEVAL_BY_SLICE.is_file():
    print(
        "Note: missing baseline eval_retrieval CSV(s); retrieval policy tables compare candidates only."
        " Expect:",
        PATH_RETRIEVAL_OVERALL,
        PATH_RETRIEVAL_BY_SLICE,
    )

diagnostic_cols = [
    c
    for c in baseline_overall.columns
    if c.startswith("CatalogCoverage@")
    or c.startswith("ILD@")
    or c.startswith("Novelty@")
    or c.startswith("PersonalizationGapVsPopularity@")
]
diagnostic_cols = sorted(set(diagnostic_cols) | {c for c in two_tower_overall.columns if c not in required})
overall_cols = ["method", *METRIC_COLS, *diagnostic_cols]

for c in diagnostic_cols:
    if c not in baseline_overall_sub.columns:
        baseline_overall_sub[c] = float("nan")
    if c not in two_tower_overall.columns:
        two_tower_overall[c] = float("nan")

compare_overall_full = pd.concat(
    [baseline_overall_sub[overall_cols], two_tower_overall[overall_cols]],
    ignore_index=True,
)
compare_by_slice_full = pd.concat(
    [
        baseline_by_slice_sub[["slice_name", "method", *METRIC_COLS]],
        two_tower_by_slice[["slice_name", "method", *METRIC_COLS, *[c for c in diagnostic_cols if c in two_tower_by_slice.columns]]],
    ],
    ignore_index=True,
)

compare_retrieval_overall_full = pd.concat(
    [
        baseline_retr_overall_sub,
        two_tower_retrieval_overall[["method", *RETRIEVAL_METRIC_COLS]],
    ],
    ignore_index=True,
)
compare_retrieval_by_slice_full = pd.concat(
    [
        baseline_retr_by_slice_sub,
        two_tower_retrieval_by_slice[["slice_name", "method", *RETRIEVAL_METRIC_COLS]],
    ],
    ignore_index=True,
)

_merge_rank_cols = [c for c in SORT_RANKING_OVERALL if c in compare_overall_full.columns]
compare_overall_full = compare_overall_full.sort_values(
    _merge_rank_cols,
    ascending=False,
).reset_index(drop=True)
compare_by_slice_full = _sort_ranking_by_slice_for_display(compare_by_slice_full)
compare_retrieval_overall_full = compare_retrieval_overall_full.sort_values(
    SORT_RETRIEVAL_OVERALL,
    ascending=False,
).reset_index(drop=True)
compare_retrieval_by_slice_full = _sort_retrieval_by_slice_for_display(compare_retrieval_by_slice_full)

print(
    "\nCombined ranking overall — primary: NDCG@K · secondary: MAP@K · tertiary: MRR "
    f"(k_final={K_FINAL}); tie-break: Hit → Precision → Recall",
)
display(compare_overall_full)

print(
    "\nCombined ranking by slice — slice A: primary NDCG@K · secondary MAP@K · tertiary MRR; "
    "slice B: primary Hit@K · secondary MRR (then NDCG@K · MAP@K)",
)
display(compare_by_slice_full)

print(
    "\nCombined retrieval overall — primary: Recall@K · secondary: Precision@K · tertiary: Hit@K "
    f"(k_retrieval={K_RETRIEVAL})",
)
display(compare_retrieval_overall_full)

print(
    "\nCombined retrieval by slice — slice A: Recall@K · Precision@K · Hit@K; "
    "slice B: Hit@K · Precision@K · Recall@K",
)
display(compare_retrieval_by_slice_full)

retrieval_eval_slice_a = (
    compare_retrieval_by_slice_full[compare_retrieval_by_slice_full["slice_name"] == "slice_a_multi_target"]
    .sort_values(["Recall@K", "Precision@K", "Hit@K"], ascending=False)
    .reset_index(drop=True)
)
retrieval_eval_slice_b = (
    compare_retrieval_by_slice_full[compare_retrieval_by_slice_full["slice_name"] == "slice_b_single_target"]
    .sort_values(["Hit@K", "Precision@K", "Recall@K"], ascending=False)
    .reset_index(drop=True)
)
ranking_eval_slice_a = (
    compare_by_slice_full[compare_by_slice_full["slice_name"] == "slice_a_multi_target"]
    .sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False)
    .reset_index(drop=True)
)
ranking_eval_slice_b = (
    compare_by_slice_full[compare_by_slice_full["slice_name"] == "slice_b_single_target"]
    .sort_values(["Hit@K", "MRR", "NDCG@K", "MAP@K"], ascending=False)
    .reset_index(drop=True)
)

print(
    "\nRetrieval (baseline+candidate) — slice A — primary: Recall@K · secondary: Precision@K · tertiary: Hit@K",
)
display(retrieval_eval_slice_a)
print(
    "\nRetrieval (baseline+candidate) — slice B — primary: Hit@K · secondary: Precision@K · tertiary: Recall@K",
)
display(retrieval_eval_slice_b)
print("\nRanking (baseline+candidate) — slice A — primary: NDCG@K · secondary: MAP@K · tertiary: MRR")
display(ranking_eval_slice_a)
print(
    "\nRanking (baseline+candidate) — slice B — primary: Hit@K · secondary: MRR · tertiary: NDCG@K (then MAP)",
)
display(ranking_eval_slice_b)

if WRITE_OUTPUTS:
    compare_overall_full.to_csv(PATH_COMPARE_OVERALL_OUT, index=False)
    compare_by_slice_full.to_csv(PATH_COMPARE_BY_SLICE_OUT, index=False)
    print("\nWrote:")
    print(" -", PATH_COMPARE_OVERALL_OUT)
    print(" -", PATH_COMPARE_BY_SLICE_OUT)
else:
    print("\nWRITE_OUTPUTS=False, no files written.")


Combined ranking overall — primary: NDCG@K · secondary: MAP@K · tertiary: MRR (k_final=10); tie-break: Hit → Precision → Recall


,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,popularity_train,0.15112,0.015184,0.146796,0.050594,0.073109,0.052221,0.034921,0.212335,5.317471,0.000000
1,two_tower_d_raw_plus_habit,0.08376,0.008424,0.077867,0.026579,0.038979,0.043440,1.000000,0.138554,9.661580,0.976607
2,two_tower_c_raw_plus_behavior,0.08536,0.008640,0.079430,0.025877,0.038836,0.042861,1.000000,0.134907,9.574771,0.973837
3,two_tower_b_raw_plus_mean_train,0.08152,0.008192,0.075468,0.025926,0.037861,0.042131,1.000000,0.147905,9.719181,0.979075
4,multi_mean_train,0.08328,0.008376,0.076479,0.025349,0.037700,0.027358,1.000000,0.146727,9.798976,0.976841
5,two_tower_a_raw_text,0.07168,0.007216,0.066606,0.024917,0.035020,0.040381,1.000000,0.160723,9.734209,0.980731
6,raw,0.07168,0.007216,0.066606,0.024917,0.035020,0.026749,1.000000,0.160723,9.838045,0.979470
7,two_tower_e_pref_structured_session,0.04664,0.004696,0.042510,0.015930,0.022508,0.028657,0.996825,0.172494,10.428898,0.994008



Combined ranking by slice — slice A: primary NDCG@K · secondary MAP@K · tertiary MRR; slice B: primary Hit@K · secondary MRR (then NDCG@K · MAP@K)


,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_a_multi_target,two_tower_c_raw_plus_behavior,0.172414,0.019034,0.070175,0.025696,0.046757,0.082511,1.000000,0.134907,9.574771,0.973837
1,slice_a_multi_target,multi_mean_train,0.188966,0.019724,0.071700,0.021733,0.044159,0.056376,NaN,NaN,NaN,NaN
2,slice_a_multi_target,two_tower_b_raw_plus_mean_train,0.171034,0.017793,0.066697,0.022662,0.042981,0.076630,1.000000,0.147905,9.719181,0.979075
3,slice_a_multi_target,two_tower_d_raw_plus_habit,0.162759,0.017103,0.061161,0.023563,0.042571,0.080748,1.000000,0.138554,9.661580,0.976607
4,slice_a_multi_target,two_tower_a_raw_text,0.143448,0.015172,0.055964,0.021677,0.038812,0.073125,1.000000,0.160723,9.734209,0.980731
5,slice_a_multi_target,raw,0.143448,0.015172,0.055964,0.021677,0.038812,0.053255,NaN,NaN,NaN,NaN
6,slice_a_multi_target,popularity_train,0.128276,0.014069,0.053728,0.019418,0.035444,0.047469,NaN,NaN,NaN,NaN
7,slice_a_multi_target,two_tower_e_pref_structured_session,0.118621,0.012414,0.047418,0.018478,0.032639,0.060171,0.996825,0.172494,10.428898,0.994008
8,slice_b_single_target,popularity_train,0.152527,0.015253,0.152527,0.052514,0.075428,0.052514,NaN,NaN,NaN,NaN
9,slice_b_single_target,two_tower_c_raw_plus_behavior,0.080000,0.008000,0.080000,0.025888,0.038348,0.040419,1.000000,0.134907,9.574771,0.973837



Combined retrieval overall — primary: Recall@K · secondary: Precision@K · tertiary: Hit@K (k_retrieval=100)


...


Ranking (baseline+candidate) — slice A — primary: NDCG@K · secondary: MAP@K · tertiary: MRR


,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_a_multi_target,two_tower_c_raw_plus_behavior,0.172414,0.019034,0.070175,0.025696,0.046757,0.082511,1.000000,0.134907,9.574771,0.973837
1,slice_a_multi_target,multi_mean_train,0.188966,0.019724,0.071700,0.021733,0.044159,0.056376,NaN,NaN,NaN,NaN
2,slice_a_multi_target,two_tower_b_raw_plus_mean_train,0.171034,0.017793,0.066697,0.022662,0.042981,0.076630,1.000000,0.147905,9.719181,0.979075
3,slice_a_multi_target,two_tower_d_raw_plus_habit,0.162759,0.017103,0.061161,0.023563,0.042571,0.080748,1.000000,0.138554,9.661580,0.976607
4,slice_a_multi_target,two_tower_a_raw_text,0.143448,0.015172,0.055964,0.021677,0.038812,0.073125,1.000000,0.160723,9.734209,0.980731
5,slice_a_multi_target,raw,0.143448,0.015172,0.055964,0.021677,0.038812,0.053255,NaN,NaN,NaN,NaN
6,slice_a_multi_target,popularity_train,0.128276,0.014069,0.053728,0.019418,0.035444,0.047469,NaN,NaN,NaN,NaN
7,slice_a_multi_target,two_tower_e_pref_structured_session,0.118621,0.012414,0.047418,0.018478,0.032639,0.060171,0.996825,0.172494,10.428898,0.994008



Ranking (baseline+candidate) — slice B — primary: Hit@K · secondary: MRR · tertiary: NDCG@K (then MAP)


,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,CatalogCoverage@10,ILD@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_b_single_target,popularity_train,0.152527,0.015253,0.152527,0.052514,0.075428,0.052514,NaN,NaN,NaN,NaN
1,slice_b_single_target,two_tower_c_raw_plus_behavior,0.080000,0.008000,0.080000,0.025888,0.038348,0.040419,1.000000,0.134907,9.574771,0.973837
2,slice_b_single_target,two_tower_d_raw_plus_habit,0.078896,0.007890,0.078896,0.026764,0.038758,0.041143,1.000000,0.138554,9.661580,0.976607
3,slice_b_single_target,multi_mean_train,0.076773,0.007677,0.076773,0.025572,0.037302,0.025572,NaN,NaN,NaN,NaN
4,slice_b_single_target,two_tower_b_raw_plus_mean_train,0.076008,0.007601,0.076008,0.026127,0.037546,0.040006,1.000000,0.147905,9.719181,0.979075
5,slice_b_single_target,two_tower_a_raw_text,0.067261,0.006726,0.067261,0.025117,0.034786,0.038365,1.000000,0.160723,9.734209,0.980731
6,slice_b_single_target,raw,0.067261,0.006726,0.067261,0.025117,0.034786,0.025117,NaN,NaN,NaN,NaN
7,slice_b_single_target,two_tower_e_pref_structured_session,0.042208,0.004221,0.042208,0.015773,0.021884,0.026717,0.996825,0.172494,10.428898,0.994008



WRITE_OUTPUTS=False, no files written.


# Visualizations

Uses **`compare_*_full`** dataframes from the merge cell above (restart kernel + run all from the top if these are missing). Baselines vs candidates share one color rule: blue vs green. Ranking panels use **`k_final`**; retrieval panels use **`k_retrieval`** from the job config.

**Layout convention:** Leaderboards read **best → worst**. **Horizontal bar plots:** best method at the **top** of the chart. **Retrieval Recall (grouped bars):** best Slice-A Recall at the **left**, decreasing to the **right**. Panel order left → right follows decision priority (e.g. retrieval: Recall → Precision → Hit).

In [41]:
from __future__ import annotations

import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _method_colors(methods: list[str]) -> list[str]:
    return ["#4662a9" if m in BASELINE_METHODS else "#1f9f6b" for m in methods]

# --- Ranking: overall (best method at top of each panel) ---
_viz_rank_cols = [c for c in SORT_RANKING_OVERALL if c in compare_overall_full.columns]
_rank = compare_overall_full.sort_values(_viz_rank_cols, ascending=False).reset_index(drop=True)
_methods = list(_rank["method"])
_colors = _method_colors(_methods)

print(
    f"Ranking viz — row sort: primary NDCG@K · secondary MAP@K · tertiary MRR "
    f"(k_final={K_FINAL}); then Hit / Precision / Recall. Bars: best rank at TOP."
)

# Prepare for horizontal bar (best method at TOP)
# Plotly plots bottom row on top, so we reverse the lists
plot_cols = ["NDCG@K", "MAP@K"]
rank_y_labels = _methods[::-1]
rank_y_indices = list(range(len(_methods)))[::-1]
color_map_reversed = _colors[::-1]

fig_rank = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=[f"{col} (ranking · k_final={K_FINAL})" for col in plot_cols])

for idx, col in enumerate(plot_cols, start=1):
    values = list(_rank[col].astype(float))[::-1]
    trace = go.Bar(
        x=values,
        y=rank_y_labels,
        orientation="h",
        marker_color=color_map_reversed,
        name=col,
        showlegend=False,
    )
    fig_rank.add_trace(trace, row=1, col=idx)
    fig_rank.update_xaxes(title_text=col, row=1, col=idx)

fig_rank.update_yaxes(tickfont=dict(size=10), title_text="method", row=1, col=1)
fig_rank.update_layout(
    title=dict(
        text="Overall ranking (merged baseline + candidates)",
        x=0.5,
        y=0.98,
        font=dict(size=14)
    ),
    height=max(320, 45 * len(_methods)),
    width=1200,
    barmode="group",
    margin=dict(t=60, b=20, l=150, r=30)
)
fig_rank.show()

Ranking viz — row sort: primary NDCG@K · secondary MAP@K · tertiary MRR (k_final=10); then Hit / Precision / Recall. Bars: best rank at TOP.


In [42]:
# Retrieval: overall (Hit / Precision / Recall @ k_retrieval), now using Plotly
_retr = compare_retrieval_overall_full.sort_values(SORT_RETRIEVAL_OVERALL, ascending=False).reset_index(drop=True)
print(
    f"Retrieval viz — row sort: primary Recall@K · secondary Precision@K · tertiary Hit@K "
    f"(k_retrieval={K_RETRIEVAL}). Bars: best at TOP. Panels LEFT→RIGHT: Recall, Precision, Hit."
)
_rm = list(_retr["method"])
_rc = _method_colors(_rm)
_retr_panel_cols = [c for c in SORT_RETRIEVAL_OVERALL if c in _retr.columns]

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare y labels (top-most is the best method—reverse order for plotly bar orientation)
rank_y_labels = _rm[::-1]
color_map_reversed = _rc[::-1]
n_methods = len(_rm)
n_panels = len(_retr_panel_cols)

fig_retr = make_subplots(
    rows=1, cols=n_panels, shared_yaxes=True,
    subplot_titles=[f"{col}" for col in _retr_panel_cols]
)

for idx, col in enumerate(_retr_panel_cols, start=1):
    values = list(_retr[col].astype(float))[::-1]
    trace = go.Bar(
        x=values,
        y=rank_y_labels,
        orientation="h",
        marker_color=color_map_reversed,
        name=col,
        showlegend=False,
    )
    fig_retr.add_trace(trace, row=1, col=idx)
    fig_retr.update_xaxes(title_text=col, row=1, col=idx)

fig_retr.update_yaxes(tickfont=dict(size=10), title_text="method", row=1, col=1)
fig_retr.update_layout(
    title=dict(
        text=f"Overall retrieval (k_retrieval={K_RETRIEVAL})",
        x=0.5,
        y=0.98,
        font=dict(size=14)
    ),
    height=max(320, 45 * n_methods),
    width=4.5 * n_panels * 100,
    barmode="group",
    margin=dict(t=60, b=20, l=150, r=30)
)
fig_retr.show()

Retrieval viz — row sort: primary Recall@K · secondary Precision@K · tertiary Hit@K (k_retrieval=100). Bars: best at TOP. Panels LEFT→RIGHT: Recall, Precision, Hit.


In [43]:
# Retrieval by slice: grouped bars for Recall@K (slice A vs B) using Plotly

import plotly.graph_objects as go
from plotly.subplots import make_subplots

_slice_df = compare_retrieval_by_slice_full[
    compare_retrieval_by_slice_full["slice_name"].isin(["slice_a_multi_target", "slice_b_single_target"])
].copy()
if _slice_df.empty:
    raise RuntimeError("compare_retrieval_by_slice_full missing slice A/B rows; re-run merge cell.")

_col_a = "slice_a_multi_target"
_col_b = "slice_b_single_target"
_pv = _slice_df.pivot_table(index="method", columns="slice_name", values="Recall@K", aggfunc="first").astype(float)
_pv = _pv.reindex(columns=[_col_a, _col_b])
_pv = _pv.dropna(how="all").sort_values([_col_a, _col_b], ascending=[False, False])
print(
    f"Recall@K by slice — x-order: best Slice A Recall at LEFT → right; "
    f"tie-break Slice B Recall (k_retrieval={K_RETRIEVAL})."
)

methods = list(_pv.index)

fig_slice = go.Figure()
fig_slice.add_trace(go.Bar(
    x=methods,
    y=_pv[_col_a],
    name="slice A (multi-target eval)",
    marker_color="#5470a9"
))
fig_slice.add_trace(go.Bar(
    x=methods,
    y=_pv[_col_b],
    name="slice B (single-target eval)",
    marker_color="#cda65b"
))

fig_slice.update_layout(
    barmode='group',
    xaxis_title="method",
    yaxis_title=f"Recall@K (k_retrieval={K_RETRIEVAL})",
    title="Retrieval Recall@K by cohort slice (best Slice A at left)",
    xaxis_tickangle=30,
    legend=dict(x=0.7, y=0.97),
    width=int(max(800, 100 * len(methods))),
    height=480,
    margin=dict(t=60, b=20, l=150, r=30)
)
fig_slice.show()

# Optional: personalization/guardrail columns on merged ranking overall (same k_personalization as job)
_gap_col = f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}"
_cov_col = f"CatalogCoverage@{K_PERSONALIZATION}"

if _gap_col in compare_overall_full.columns:
    _vcols = [c for c in SORT_RANKING_OVERALL if c in compare_overall_full.columns]
    dd = compare_overall_full.sort_values(_vcols, ascending=False).reset_index(drop=True)
    print(
        f"Guardrail viz — best at TOP (same row order as ranking chart; sort key {_vcols[0]} …). "
        "Left: personalization gap vs popularity; right: catalog coverage."
    )
    _dm = list(dd["method"])
    _colors = _method_colors(_dm)
    _gp_y = list(reversed(_dm))  # Best at top

    fig_gp = make_subplots(
        rows=1, cols=2, shared_yaxes=True,
        subplot_titles=[_gap_col, _cov_col if _cov_col in dd.columns else ""]
    )

    fig_gp.add_trace(
        go.Bar(
            y=_gp_y,
            x=dd[_gap_col][::-1],
            orientation="h",
            name=_gap_col,
            marker_color=_colors[::-1]
        ),
        row=1, col=1
    )

    if _cov_col in dd.columns:
        fig_gp.add_trace(
            go.Bar(
                y=_gp_y,
                x=dd[_cov_col][::-1],
                orientation="h",
                name=_cov_col,
                marker_color=_colors[::-1]
            ),
            row=1, col=2
        )

    fig_gp.update_xaxes(title_text="gap (higher ⇒ more personalized vs popularity)", row=1, col=1)
    if _cov_col in dd.columns:
        fig_gp.update_xaxes(title_text="fraction of catalog covered (short lists)", row=1, col=2)
    fig_gp.update_yaxes(title_text="method", row=1, col=1, tickfont=dict(size=9))
    fig_gp.update_layout(
        height=max(400, int(32 * len(_gp_y))),
        width=850,
        title_text=f"Short-list diagnostics (top k_personalization={K_PERSONALIZATION})",
        barmode="group",
        margin=dict(t=60, b=20, l=150, r=30),
        showlegend=False,
    )
    fig_gp.show()
else:
    print(f"Skipping guardrail plots: {_gap_col!r} not in compare_overall_full (baseline CSV lacks diagnostics?).")

Recall@K by slice — x-order: best Slice A Recall at LEFT → right; tie-break Slice B Recall (k_retrieval=100).


Guardrail viz — best at TOP (same row order as ranking chart; sort key NDCG@K …). Left: personalization gap vs popularity; right: catalog coverage.
